## Post Process Filtering 

In [ ]:
%load_ext autoreload
%autoreload 2

import pylupnt as pnt
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## load file

In [ ]:
satid = 0
dt = 1.0

# case_names = ["mc_if_code_tdcpL1_6", "mc_if_code_tdcpL1_7",
#               "mc_if_code_tdcpL1_8", "mc_if_code_tdcpL1_9",
#               "mc_if_code"]
# # # case_names = ["mc_if_code_tdcpL1_8", "mc_if_code"]

case_names = [
    "mc_if_code",
    "mc_if_tdcp_fault10_proc8",  # with fix
    "mc_if_tdcp_fault10_proc9_nofix",  # without fix
    #   "mc_if_tdcp_fault0_proc8",      # with fix "mc_if_code_tdcpL1_8",
    #   "mc_if_tdcp_fault0_proc8_nofix",   # without fix
    #   "mc_if_tdcp_fault0_proc9_nofix"  # without fix
]

norbit = 6
dt = 1
dt_rt = 120

str_orbitdt = f"norbit_{norbit}_dt_{dt}s_dtrt_{dt_rt}s"

outdirs = {}
for case_name in case_names:
    outdir = (
        pnt.get_output_dir()
        / "Plasmasphere_Delay_Filtering"
        / "v0"
        / "filter"
        / str_orbitdt
        / case_name
    )
    outdirs[case_name] = outdir

In [ ]:
# Load H5 file
import h5py

# list all files in outdir
h5files = {}
mcidxs = {}

for case_name in case_names:
    print("Case:", case_name)
    outdir = outdirs[case_name]
    h5files[case_name] = []
    mcidxs[case_name] = []
    for i, f in enumerate(outdir.glob("*")):
        # if end in .h5
        if f.suffix == ".h5":
            print("Loading:", f)
            h5files[case_name].append(h5py.File(f, "r"))
            # filter_case_mc_if_code_tdcpL1_6_dt1_mc3.h5
            mcidx = int(f.stem.split("mc")[-1])
            mcidxs[case_name].append(mcidx)

# sort h5files by mcidxs
for case_name in case_names:
    mcidxs_sorted = np.argsort(mcidxs[case_name])
    h5files[case_name] = [h5files[case_name][i] for i in mcidxs_sorted]
    mcidxs[case_name] = [mcidxs[case_name][i] for i in mcidxs_sorted]

# List all groups
print("Keys: %s" % h5files[case_names[0]][0].keys())

In [ ]:
# Compute Errors
from src_py.filter_postprocess_old import compute_od_errors_old

for case_name in case_names:
    print("Case: ", case_name)
    sise_pos_mat, sise_vel_mat = compute_od_errors_old(
        h5files[case_name], start_ratio=5 / 6, end_ratio=6 / 6
    )
    print(" ")
    print(" ")

In [ ]:
from src_py.filter_postprocess import plot_errors

plot_case = case_names[1]
plot_mc = 0

ylims = {
    "pos": [-50, 50],
    "pos_norm": [0, 50],
    "vel": [-1, 1],
    "vel_norm": [0, 10],
    "clkb": [-50, 50],
    "clkd": [-0.005, 0.005],
    "clkdd": [-1e-6, 1e-6],
    "srp": [-1e-3, 1e-3],
}

plot_errors(
    h5files[plot_case][plot_mc], plot_inv=10, ylims=ylims, use_rtn=True, n_orbits=norbit
)

In [ ]:
coeff = pnt.GM_MOON / pnt.C

e = 0.691
a = 11315.9365e3  # 10,000 km
r0 = a * (1 - e)  # Perilune

delta_x = 1000.0  # 10 meters
delta_v = 1e-3  # 1 mm/s

v0 = np.sqrt(pnt.GM_MOON * (2 / r0 - 1 / a))  # Circular orbit velocity at r0
time_delay_error = coeff * (1 / r0 - 1 / (r0 + delta_x))
time_delay_error_vel = ((v0 * v0) - (v0 - delta_v) * (v0 - delta_v)) / (2 * pnt.C)

print(
    "Time delay error for delta_x =",
    delta_x,
    "m at r =",
    r0,
    "m:",
    time_delay_error,
    "m",
)
print(
    "Time delay error for delta_v =",
    delta_v,
    "m/s at r =",
    r0,
    "m:",
    time_delay_error_vel,
    "m",
)
print("GM_MOON / C =", coeff, "m")

dt = 200 * 3600
print(
    "Accum Delay:",
    (time_delay_error + time_delay_error_vel) * (dt * dt / 2),
    "m over",
    dt,
    "s",
)

In [ ]:
f_L1 = 154
# 1575.42;  # L1 frequency in MHz
f_L5 = 115
# 1176.45;  # L5 frequency in MHz
iono_factor_L1 = f_L1 * f_L1 / (f_L1 * f_L1 - f_L5 * f_L5)
iono_factor_L5 = f_L5 * f_L5 / (f_L1 * f_L1 - f_L5 * f_L5)

print("Iono factor L1:", iono_factor_L1)
print("Iono factor L5:", iono_factor_L5)

In [ ]:
L1_lambda = pnt.C / 1575.42e6
L5_lambda = pnt.C / 1176.45e6
L1L5_lambda = iono_factor_L1 * L1_lambda - iono_factor_L5 * L5_lambda

print("L1 wavelength:", L1_lambda, "m")
print("L5 wavelength:", L5_lambda, "m")
print("L1-L5 wavelength:", L1L5_lambda, "m")